# Notebook 17 — Feature Engineering
### Heterogeneous Treatment Effects in Mortgage Lending
### Extending HMDA Racial Disparity Analysis with Causal Forests and Double ML

**Author:** Rajveer Singh Pall  
**Institution:** Gyan Ganga Institute of Technology and Sciences  

---

Builds the full covariate matrix for CATE estimation by augmenting the
processed panel files (from Repo 1) with additional HMDA variables:
loan_purpose, property_type, occupancy_type, loan_type, and lender size tier.

**INPUT:**
- `data/processed/panel_YYYY.csv` — cleaned panels from Repo 1 (NB01)
- `data/hmda_YYYY.csv` — raw HMDA files (for additional columns)

**OUTPUT:**
- `data/features_panel.parquet` — full feature matrix, all years combined
- `outputs/tables/nb17_feature_summary.csv` — feature completeness report

**MEMORY STRATEGY:**
- Raw HMDA files read in 500K-row chunks (same as NB13)
- Only Black (race=3) and White (race=5) applicants kept
- Final parquet compressed with snappy — ~3-5GB on disk, fast to read

**RUNTIME:** ~20-40 minutes on i7-13650HX

In [1]:
# ============================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────
# NOTE: Adjust BASE_DIR if your folder structure differs
BASE_DIR       = Path("D:/CATE-HMDA-Heterogeneous-Effects")
DATA_DIR       = BASE_DIR / "data"
PROCESSED_DIR  = DATA_DIR / "processed"
TABLES_DIR     = BASE_DIR / "outputs" / "tables"
FIGURES_DIR    = BASE_DIR / "outputs" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

YEARS      = [2020, 2021, 2022, 2023, 2024]
BLACK_CODE = 3
WHITE_CODE = 5
CHUNK_SIZE = 500_000

print("✅ Configuration loaded")
print(f"   Base directory : {BASE_DIR}")
print(f"   Data directory : {DATA_DIR}")
print(f"   Processed dir  : {PROCESSED_DIR}")

# Verify key paths exist
for yr in YEARS:
    p = PROCESSED_DIR / f"panel_{yr}.csv"
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"   panel_{yr}.csv : {status}")

print()
for yr in YEARS:
    p = DATA_DIR / f"hmda_{yr}.csv"
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"   hmda_{yr}.csv  : {status}")

✅ Configuration loaded
   Base directory : D:\CATE-HMDA-Heterogeneous-Effects
   Data directory : D:\CATE-HMDA-Heterogeneous-Effects\data
   Processed dir  : D:\CATE-HMDA-Heterogeneous-Effects\data\processed
   panel_2020.csv : ✅
   panel_2021.csv : ✅
   panel_2022.csv : ✅
   panel_2023.csv : ✅
   panel_2024.csv : ✅

   hmda_2020.csv  : ✅
   hmda_2021.csv  : ✅
   hmda_2022.csv  : ✅
   hmda_2023.csv  : ✅
   hmda_2024.csv  : ✅


In [2]:
# ============================================================================
# CELL 2 — INSPECT RAW HMDA COLUMNS
# Reads only the header row to confirm which additional columns are available.
# Run this once to verify column names before the full extraction.
# ============================================================================

print("="*70)
print("INSPECTING RAW HMDA COLUMN NAMES (2024 file, header only)")
print("="*70)

sample = pd.read_csv(DATA_DIR / "hmda_2024.csv", nrows=5)
print(f"\nTotal columns in raw HMDA: {len(sample.columns)}")
print("\nAll columns:")
for i, col in enumerate(sample.columns):
    print(f"  {i:3d}  {col}")

# The columns we need beyond what's in the processed panel:
EXTRA_COLS_NEEDED = [
    'loan_purpose',      # 1=purchase, 31=refi rate-term, 32=refi cash-out, 4=home improvement
    'property_type',     # NOTE: renamed to 'construction_method' in newer HMDA
    'occupancy_type',    # 1=principal, 2=second, 3=investment
    'loan_type',         # 1=conventional, 2=FHA, 3=VA, 4=USDA
    'derived_loan_product_type',  # may exist in newer years
]

print("\n" + "="*70)
print("CHECKING AVAILABILITY OF TARGET COLUMNS")
print("="*70)
for col in EXTRA_COLS_NEEDED:
    status = "✅ FOUND" if col in sample.columns else "❌ NOT FOUND — check alternate name"
    print(f"  {col}: {status}")

INSPECTING RAW HMDA COLUMN NAMES (2024 file, header only)

Total columns in raw HMDA: 99

All columns:
    0  activity_year
    1  lei
    2  derived_msa_md
    3  state_code
    4  county_code
    5  census_tract
    6  conforming_loan_limit
    7  derived_loan_product_type
    8  derived_dwelling_category
    9  derived_ethnicity
   10  derived_race
   11  derived_sex
   12  action_taken
   13  purchaser_type
   14  preapproval
   15  loan_type
   16  loan_purpose
   17  lien_status
   18  reverse_mortgage
   19  open_end_line_of_credit
   20  business_or_commercial_purpose
   21  loan_amount
   22  combined_loan_to_value_ratio
   23  interest_rate
   24  rate_spread
   25  hoepa_status
   26  total_loan_costs
   27  total_points_and_fees
   28  origination_charges
   29  discount_points
   30  lender_credits
   31  loan_term
   32  prepayment_penalty_term
   33  intro_rate_period
   34  negative_amortization
   35  interest_only_payment
   36  balloon_payment
   37  other_nonamortiz

In [3]:
# ============================================================================
# CELL 3 — EXTRACT COLUMNS FROM RAW HMDA — POLARS OPTIMIZED
#
# WHY POLARS INSTEAD OF PANDAS CHUNKED READING:
#   - Polars is written in Rust with true multi-threading
#   - Uses ALL 14 cores of your i7-13650HX simultaneously
#   - Reads CSV ~6-8x faster than pandas chunked approach
#   - No chunk loop overhead — reads entire file in one parallel pass
#   - Expected time: ~1 min total vs ~6 min with pandas chunks
#
# NOTE ON GPU: CSV parsing cannot use GPU (RTX 4060 idle here — that's fine,
#   GPU will be used in later notebooks for LightGBM if needed).
#   The i7-13650HX's 14 cores are the right tool for this task.
#
# FALLBACK: If polars fails for any reason, code falls back to pandas chunks
# ============================================================================

import time

# Try to import polars — if not installed, fall back to pandas
try:
    import polars as pl
    USE_POLARS = True
    print(f"✅ Polars {pl.__version__} loaded — fast multi-threaded mode")
except ImportError:
    USE_POLARS = False
    print("⚠️  Polars not installed — using pandas fallback")
    print("   To get ~6x speedup: pip install polars")

print("=" * 70)
print("EXTRACTING COLUMNS FROM RAW HMDA")
print("=" * 70)

RAW_COLS = [
    # Keys for merging
    'lei',
    'applicant_race_1',
    'action_taken',
    'income',
    'loan_amount',
    'property_value',
    # Loan characteristics
    'loan_purpose',
    'construction_method',
    'occupancy_type',
    'loan_type',
    # Identification improvements
    'debt_to_income_ratio',
    'combined_loan_to_value_ratio',
    'applicant_age',
    'applicant_age_above_62',
    # Mechanism channels
    'aus_1',
    'denial_reason_1',
    # Geographic
    'state_code',
    'tract_minority_population_percent',
    'ffiec_msa_md_median_family_income',
]

def get_available_cols(filepath, desired_cols):
    """Returns only columns that actually exist in this file."""
    header = pd.read_csv(filepath, nrows=0).columns.tolist()
    available = [c for c in desired_cols if c in header]
    missing   = [c for c in desired_cols if c not in header]
    if missing:
        print(f"   ⚠️  Not in this file (skipped): {missing}")
    return available


def extract_year_polars(year, raw_path, cache_path):
    """
    Fast extraction using Polars.
    Reads the entire file in one multi-threaded pass, filters, saves to parquet.
    Uses lazy evaluation — only materialises rows that pass the filter.
    """
    cols_available = get_available_cols(raw_path, RAW_COLS)

    t0 = time.time()

    # Polars lazy scan — reads only needed columns, filters in parallel
    # infer_schema_length=0 reads all columns as strings first (safe for mixed types)
    lf = (
        pl.scan_csv(
            str(raw_path),
            infer_schema_length=10000,
            null_values=["", "NA", "N/A", "Exempt"],
            ignore_errors=True,
        )
        .select([c for c in cols_available if c in pl.scan_csv(str(raw_path), n_rows=1).columns])
    )

    # Cast key columns to correct types before filtering
    # applicant_race_1 and action_taken may be read as float due to NaNs
    lf = lf.with_columns([
        pl.col("applicant_race_1").cast(pl.Float64, strict=False),
        pl.col("action_taken").cast(pl.Float64, strict=False),
    ])

    # Filter: Black (3) or White (5) applicants, originated (1) or denied (3)
    lf = lf.filter(
        pl.col("applicant_race_1").is_in([BLACK_CODE, WHITE_CODE]) &
        pl.col("action_taken").is_in([1.0, 3.0])
    )

    # Add year column
    lf = lf.with_columns(pl.lit(year).alias("year"))

    # Collect (execute the query — this is where the actual work happens)
    df_year_pl = lf.collect()

    elapsed = time.time() - t0

    total_rows = pl.scan_csv(str(raw_path), n_rows=1).select(pl.count()).collect().item()
    # We can't get exact total easily with lazy scan, so estimate
    kept = len(df_year_pl)

    print(f"   Kept     : {kept:>12,} rows in {elapsed:.1f}s  "
          f"({elapsed/60:.1f} min)")
    print(f"   Columns  : {df_year_pl.columns}")

    # Convert to pandas for compatibility with rest of notebook
    df_year_pd = df_year_pl.to_pandas()

    # Ensure lei stays as string
    if 'lei' in df_year_pd.columns:
        df_year_pd['lei'] = df_year_pd['lei'].astype(str)

    # Save parquet via pandas (compatible with downstream pandas notebooks)
    df_year_pd.to_parquet(cache_path, compression='snappy', index=False)
    print(f"   Saved    : {cache_path.name}  ({cache_path.stat().st_size/1e6:.0f} MB)")

    return df_year_pd


def extract_year_pandas(year, raw_path, cache_path):
    """
    Fallback: original pandas chunked approach with larger 2M chunk size.
    2M chunks instead of 500K = ~4x fewer iterations = ~4x faster than original.
    """
    cols_available = get_available_cols(raw_path, RAW_COLS)
    t0 = time.time()

    chunks     = []
    total_rows = 0
    kept_rows  = 0

    reader = pd.read_csv(
        raw_path,
        usecols=cols_available,
        chunksize=2_000_000,          # 2M rows per chunk — 4x larger than before
        dtype={'lei': str, 'state_code': str},
        low_memory=False
    )

    for chunk in tqdm(reader, desc=f"  {year}"):
        total_rows += len(chunk)
        mask = (
            chunk['applicant_race_1'].isin([BLACK_CODE, WHITE_CODE]) &
            chunk['action_taken'].isin([1, 3])
        )
        chunk = chunk[mask].copy()
        kept_rows += len(chunk)
        if len(chunk) > 0:
            chunk['year'] = year
            chunks.append(chunk)

    df_year = pd.concat(chunks, ignore_index=True)
    df_year.to_parquet(cache_path, compression='snappy', index=False)

    elapsed = time.time() - t0
    print(f"   Scanned  : {total_rows:>12,} rows")
    print(f"   Kept     : {kept_rows:>12,} ({100*kept_rows/total_rows:.1f}%) in {elapsed:.1f}s")
    return df_year


# ── MAIN EXTRACTION LOOP ───────────────────────────────────────────────────
extras_by_year = {}
total_start    = time.time()

for year in YEARS:
    raw_path   = DATA_DIR / f'hmda_{year}.csv'
    cache_path = DATA_DIR / f'extras_{year}.parquet'

    print(f"\n{'─'*70}")
    print(f"YEAR {year}")
    print(f"{'─'*70}")

    # Skip if already cached from a previous run
    if cache_path.exists():
        print(f"  Cache found → loading {cache_path.name}")
        extras_by_year[year] = pd.read_parquet(cache_path)
        print(f"  {len(extras_by_year[year]):,} rows loaded from cache")
        continue

    if not raw_path.exists():
        print(f"  ❌ File not found: {raw_path}")
        print(f"     Check your symlink or copy the file to data/")
        continue

    file_size_gb = raw_path.stat().st_size / 1e9
    print(f"  File size: {file_size_gb:.1f} GB")

    if USE_POLARS:
        try:
            extras_by_year[year] = extract_year_polars(year, raw_path, cache_path)
        except Exception as e:
            print(f"  ⚠️  Polars failed ({e}), falling back to pandas...")
            extras_by_year[year] = extract_year_pandas(year, raw_path, cache_path)
    else:
        extras_by_year[year] = extract_year_pandas(year, raw_path, cache_path)

    gc.collect()

total_elapsed = time.time() - total_start
print(f"\n{'='*70}")
print(f"✅ Extraction complete — all years done in {total_elapsed:.0f}s "
      f"({total_elapsed/60:.1f} min)")
print(f"{'='*70}")

# ── QUICK SPOT CHECKS ──────────────────────────────────────────────────────
print("\nSpot checks across years:")
for year in YEARS:
    if year in extras_by_year:
        df_y  = extras_by_year[year]
        n     = len(df_y)
        b_pct = 100 * (df_y['applicant_race_1'] == BLACK_CODE).mean()
        has_dti = 'debt_to_income_ratio' in df_y.columns
        print(f"  {year}: {n:>9,} rows | Black: {b_pct:.1f}% | DTI col: {'✅' if has_dti else '❌'}")

print("\nDTI value counts (2024, top 10):")
if 2024 in extras_by_year and 'debt_to_income_ratio' in extras_by_year[2024].columns:
    print(extras_by_year[2024]['debt_to_income_ratio'].value_counts().head(10).to_string())

✅ Polars 1.39.0 loaded — fast multi-threaded mode
EXTRACTING COLUMNS FROM RAW HMDA

──────────────────────────────────────────────────────────────────────
YEAR 2020
──────────────────────────────────────────────────────────────────────
  Cache found → loading extras_2020.parquet
  13,341,411 rows loaded from cache

──────────────────────────────────────────────────────────────────────
YEAR 2021
──────────────────────────────────────────────────────────────────────
  Cache found → loading extras_2021.parquet
  13,368,504 rows loaded from cache

──────────────────────────────────────────────────────────────────────
YEAR 2022
──────────────────────────────────────────────────────────────────────
  Cache found → loading extras_2022.parquet
  8,111,843 rows loaded from cache

──────────────────────────────────────────────────────────────────────
YEAR 2023
──────────────────────────────────────────────────────────────────────
  Cache found → loading extras_2023.parquet
  5,817,165 rows loade

In [4]:
# ============================================================================
# CELL 4 — MERGE + SAVE YEAR BY YEAR (DTYPE-SAFE VERSION)
# ============================================================================

import gc
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR      = Path("D:/CATE-HMDA-Heterogeneous-Effects")
DATA_DIR      = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
YEARS         = [2020, 2021, 2022, 2023, 2024]

MERGE_KEY = ['lei', 'year', 'applicant_race_1', 'income', 'loan_amount']

ALL_EXTRA_COLS = [
    'loan_purpose', 'construction_method', 'occupancy_type', 'loan_type',
    'debt_to_income_ratio', 'combined_loan_to_value_ratio',
    'applicant_age', 'applicant_age_above_62',
    'aus_1', 'denial_reason_1',
    'state_code', 'tract_minority_population_percent',
    'ffiec_msa_md_median_family_income',
]

# Columns we ALWAYS want to keep — drop everything else from panel
# This prevents mystery columns like rate_spread causing Arrow errors
KEEP_COLS = [
    # Identifiers
    'lei', 'year', 'applicant_race_1', 'black',
    # Outcome + core financials
    'approved', 'income', 'loan_amount', 'property_value', 'ltv',
    # All extra cols we explicitly pulled
] + ALL_EXTRA_COLS

def sanitize_for_parquet(df):
    """
    Fixes all dtype issues that cause PyArrow to fail on to_parquet().
    - Object columns with mixed float/string → cast to string, then
      numeric strings stay numeric, garbage becomes NaN
    - Nullable integer types (Int32, Int64) → standard numpy int32
    - Float64 → float32
    - Int64 → int32
    """
    for col in df.columns:
        dtype = df[col].dtype

        # Pandas nullable integer types (Int32, Int64 with capital I)
        # These come from read_parquet with dtype_backend='numpy_nullable'
        if hasattr(dtype, 'numpy_dtype'):
            try:
                df[col] = df[col].astype(float).astype(np.float32)
            except Exception:
                df[col] = df[col].astype(str)
            continue

        if dtype == object:
            # Try converting to numeric first
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notna().mean() > 0.5:
                # Mostly numeric — save as float32
                df[col] = converted.astype(np.float32)
            else:
                # Mostly strings — keep as string, replace any non-string
                df[col] = df[col].astype(str).replace('nan', None)

        elif dtype == np.float64:
            df[col] = df[col].astype(np.float32)

        elif dtype == np.int64:
            df[col] = df[col].astype(np.int32)

    return df


# Clear memory
for _v in ['extras_by_year', 'all_years', 'df', 'merged', 'panel', 'extras']:
    if _v in globals():
        del globals()[_v]
gc.collect()
print("✅ Memory cleared")
print("=" * 70)

merged_paths = []

for year in YEARS:
    out_path = DATA_DIR / f'merged_{year}.parquet'
    merged_paths.append(out_path)

    if out_path.exists():
        print(f"{year}: ✅ already saved ({out_path.stat().st_size/1e6:.0f} MB) — skipping")
        continue

    print(f"\n{'─'*60}")
    print(f"YEAR {year}")

    # ── Load panel ─────────────────────────────────────────────────────────
    panel = pd.read_csv(PROCESSED_DIR / f'panel_{year}.csv', dtype={'lei': str})
    panel['year'] = year

    # Keep only the columns we want — drops rate_spread and other junk
    cols_to_keep_panel = [c for c in KEEP_COLS if c in panel.columns]
    panel = panel[cols_to_keep_panel]
    print(f"  Panel  : {len(panel):>8,} rows  cols={list(panel.columns)}")

    # ── Load extras (only needed columns) ──────────────────────────────────
    cols_to_add = [
        c for c in ALL_EXTRA_COLS
        if c not in set(panel.columns) and c not in MERGE_KEY
    ]

    if cols_to_add:
        extras_slim = pd.read_parquet(
            DATA_DIR / f'extras_{year}.parquet',
            columns=MERGE_KEY + cols_to_add
        )
        extras_slim['lei'] = extras_slim['lei'].astype(str)
        extras_slim = extras_slim.drop_duplicates(subset=MERGE_KEY, keep='first')
        print(f"  Extras : {len(extras_slim):>8,} rows  adding={cols_to_add}")

        merged = panel.merge(extras_slim, on=MERGE_KEY, how='left')
        del extras_slim, panel
    else:
        print(f"  No new cols to add")
        merged = panel
        del panel

    gc.collect()

    # ── Sanitize all dtypes before saving ──────────────────────────────────
    print(f"  Sanitizing dtypes...")
    merged = sanitize_for_parquet(merged)

    # Final dtype report
    obj_cols = [c for c in merged.columns if merged[c].dtype == object]
    if obj_cols:
        print(f"  String cols (expected): {obj_cols}")

    # ── Save ───────────────────────────────────────────────────────────────
    merged.to_parquet(out_path, compression='snappy', index=False)
    print(f"  Saved  : {out_path.name}  ({out_path.stat().st_size/1e6:.0f} MB)")
    print(f"  Shape  : {merged.shape}  RAM freed ✅")

    del merged
    gc.collect()

# ── Combine with Polars streaming ──────────────────────────────────────────
print(f"\n{'='*70}")
print("COMBINING WITH POLARS STREAMING")

final_path = DATA_DIR / 'all_years_merged.parquet'

if final_path.exists():
    print(f"Already exists → {final_path.name}  ({final_path.stat().st_size/1e9:.2f} GB)")
else:
    pl.scan_parquet([str(p) for p in merged_paths]).sink_parquet(
        str(final_path), compression='snappy'
    )
    print(f"✅ Saved: {final_path.name}  ({final_path.stat().st_size/1e9:.2f} GB)")

# ── Verify ─────────────────────────────────────────────────────────────────
n_rows = pl.scan_parquet(str(final_path)).select(pl.len()).collect().item()
cols   = pl.scan_parquet(str(final_path)).columns
print(f"\nTotal rows : {n_rows:,}")
print(f"Columns    : {cols}")
print(f"\n✅ Cell 4 complete → proceed to Cell 5")

✅ Memory cleared
2020: ✅ already saved (164 MB) — skipping
2021: ✅ already saved (169 MB) — skipping
2022: ✅ already saved (108 MB) — skipping
2023: ✅ already saved (77 MB) — skipping
2024: ✅ already saved (80 MB) — skipping

COMBINING WITH POLARS STREAMING
Already exists → all_years_merged.parquet  (0.82 GB)

Total rows : 43,441,950
Columns    : ['lei', 'year', 'applicant_race_1', 'black', 'approved', 'income', 'loan_amount', 'property_value', 'ltv', 'loan_purpose', 'state_code', 'construction_method', 'occupancy_type', 'loan_type', 'debt_to_income_ratio', 'combined_loan_to_value_ratio', 'applicant_age', 'applicant_age_above_62', 'aus_1', 'denial_reason_1', 'tract_minority_population_percent', 'ffiec_msa_md_median_family_income']

✅ Cell 4 complete → proceed to Cell 5


In [12]:
# ============================================================================
# CELL 4B — PATCH tract_minority_pct INTO MERGED FILE
# This column was dropped before Cell 5 encoded it. We re-extract it here
# from the extras parquets and add it to all_years_merged.parquet.
# ============================================================================

import polars as pl
import gc

print("Patching tract_minority_pct into all_years_merged.parquet...")

patched_path = DATA_DIR / 'all_years_merged_patched.parquet'

frames = []
for year in YEARS:
    extras = pl.read_parquet(
        DATA_DIR / f'extras_{year}.parquet',
        columns=['lei', 'year', 'applicant_race_1', 'income',
                 'loan_amount', 'tract_minority_population_percent']
    )
    extras = extras.rename(
        {'tract_minority_population_percent': 'tract_minority_pct_raw'}
    )
    frames.append(extras)

extras_all = pl.concat(frames)
del frames
gc.collect()

# Load main merged file and join
main = pl.scan_parquet(str(DATA_DIR / 'all_years_merged.parquet'))
extras_lf = extras_all.lazy().with_columns(
    pl.col('tract_minority_pct_raw').cast(pl.Float32)
)

patched = main.join(
    extras_lf,
    on=['lei', 'year', 'applicant_race_1', 'income', 'loan_amount'],
    how='left'
).rename({'tract_minority_pct_raw': 'tract_minority_pct_fixed'})

patched.sink_parquet(str(patched_path), compression='snappy')
del extras_all
gc.collect()

# Overwrite original
import shutil
shutil.move(str(patched_path), str(DATA_DIR / 'all_years_merged.parquet'))

# Verify
n = pl.scan_parquet(str(DATA_DIR / 'all_years_merged.parquet')).select(pl.len()).collect().item()
med = pl.scan_parquet(str(DATA_DIR / 'all_years_merged.parquet')).select(
    pl.col('tract_minority_pct_fixed').median()
).collect().item()
print(f"✅ Patched. Rows: {n:,}  |  Tract minority pct median: {med:.1f}%")
print("Now re-run Cell 5 onwards.")

Patching tract_minority_pct into all_years_merged.parquet...
✅ Patched. Rows: 747,495,075  |  Tract minority pct median: 21.8%
Now re-run Cell 5 onwards.


In [ ]:
# ============================================================================
# CELL 5 — FEATURE CONSTRUCTION (UPDATED)
#
# ORIGINAL features:
# 1. LTV (verify + filter)
# 2. Loan purpose dummies
# 3. Loan type dummies
# 4. Occupancy dummies
# 5. Lender size tier
# 6. Log transforms
# 7. PMI boundary indicator
# 8. Post-tightening indicator
#
# ADDED (from column audit in Cell 2):
# 9.  DTI encoding       — addresses Repo 1 missing-control limitation
# 10. Construction method — site-built vs manufactured (was "property_type")
# 11. AUS type           — automated vs manual underwriting
# 12. Neighbourhood composition — tract minority population
# 13. Applicant age encoding
# ============================================================================
import polars as pl
df = pl.read_parquet(str(DATA_DIR / 'all_years_merged.parquet')).to_pandas()
print(f"Loaded: {len(df):,} rows × {len(df.columns)} cols")
print(f"RAM: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print("="*70)
print("FEATURE CONSTRUCTION")
print("="*70)

# ── 1. Verify LTV ──────────────────────────────────────────────────────────
if 'ltv' not in df.columns:
    df['ltv'] = df['loan_amount'] / df['property_value'] * 100
    print("   ltv computed from loan_amount / property_value")
else:
    print("   ltv already present")

# Clip LTV to valid range (mirrors Repo 1 filter)
df = df[(df['ltv'] >= 1) & (df['ltv'] <= 200)].copy()
print(f"   After LTV filter: {len(df):,} rows")

# ── 2. Loan purpose dummies ────────────────────────────────────────────────
# HMDA codes: 1=purchase, 31=refi rate-term, 32=refi cash-out, 4=home improvement
if 'loan_purpose' in df.columns:
    df['purpose_purchase']  = (df['loan_purpose'] == 1).astype(np.int8)
    df['purpose_refi']      = (df['loan_purpose'].isin([31, 32])).astype(np.int8)
    df['purpose_homeimprv'] = (df['loan_purpose'] == 4).astype(np.int8)
    print("   Loan purpose dummies: ✅")
    print(f"     Purchase : {df['purpose_purchase'].sum():>9,} ({100*df['purpose_purchase'].mean():.1f}%)")
    print(f"     Refi     : {df['purpose_refi'].sum():>9,} ({100*df['purpose_refi'].mean():.1f}%)")
    print(f"     HomeImprv: {df['purpose_homeimprv'].sum():>9,} ({100*df['purpose_homeimprv'].mean():.1f}%)")
else:
    print("   ⚠️  loan_purpose not found — dummies set to 0")
    df['purpose_purchase']  = np.int8(0)
    df['purpose_refi']      = np.int8(0)
    df['purpose_homeimprv'] = np.int8(0)

# ── 3. Loan type dummies ───────────────────────────────────────────────────
# HMDA codes: 1=conventional, 2=FHA, 3=VA, 4=USDA
if 'loan_type' in df.columns:
    df['type_conventional'] = (df['loan_type'] == 1).astype(np.int8)
    df['type_fha']          = (df['loan_type'] == 2).astype(np.int8)
    df['type_va']           = (df['loan_type'] == 3).astype(np.int8)
    df['type_usda']         = (df['loan_type'] == 4).astype(np.int8)
    print("   Loan type dummies: ✅")
else:
    print("   ⚠️  loan_type not found — dummies set to 0")
    for col in ['type_conventional', 'type_fha', 'type_va', 'type_usda']:
        df[col] = np.int8(0)

# ── 4. Occupancy dummies ───────────────────────────────────────────────────
# HMDA codes: 1=principal residence, 2=second home, 3=investment property
if 'occupancy_type' in df.columns:
    df['occ_primary']    = (df['occupancy_type'] == 1).astype(np.int8)
    df['occ_second']     = (df['occupancy_type'] == 2).astype(np.int8)
    df['occ_investment'] = (df['occupancy_type'] == 3).astype(np.int8)
    print("   Occupancy dummies: ✅")
else:
    print("   ⚠️  occupancy_type not found — dummies set to 0")
    for col in ['occ_primary', 'occ_second', 'occ_investment']:
        df[col] = np.int8(0)

# ── 5. Lender size tier ────────────────────────────────────────────────────
print("\n   Computing lender size tiers...")
lender_volume = df.groupby('lei')['approved'].count().rename('lender_total_apps')
df = df.merge(lender_volume.reset_index(), on='lei', how='left')

# Tercile cut: small (<500 apps), mid (500-5000), large (>5000)
df['lender_small']  = (df['lender_total_apps'] < 500).astype(np.int8)
df['lender_mid']    = ((df['lender_total_apps'] >= 500) & (df['lender_total_apps'] < 5000)).astype(np.int8)
df['lender_large']  = (df['lender_total_apps'] >= 5000).astype(np.int8)
print(f"     Small lenders (<500 apps)   : {df['lender_small'].sum():>9,} applications")
print(f"     Mid lenders (500-5000)      : {df['lender_mid'].sum():>9,} applications")
print(f"     Large lenders (>5000 apps)  : {df['lender_large'].sum():>9,} applications")

# ── 6. Log transforms ─────────────────────────────────────────────────────
df['log_income']       = np.log1p(df['income'].clip(lower=0))
df['log_loan_amount']  = np.log1p(df['loan_amount'].clip(lower=0))
df['log_property_val'] = np.log1p(df['property_value'].clip(lower=0))
print("   Log transforms: ✅")

# ── 7. PMI boundary indicator ──────────────────────────────────────────────
# Flag applications near the 80% LTV threshold (within ±5pp)
df['near_pmi_threshold'] = ((df['ltv'] >= 75) & (df['ltv'] <= 85)).astype(np.int8)
df['above_pmi']          = (df['ltv'] > 80).astype(np.int8)
print(f"   Near PMI threshold (75-85% LTV): {df['near_pmi_threshold'].sum():>9,} applications")

# ── 8. Post-tightening indicator ──────────────────────────────────────────
df['post_tightening'] = (df['year'] >= 2022).astype(np.int8)

# ==========================================================================
# ── 9. DTI ENCODING (NEW) ─────────────────────────────────────────────────
# HMDA reports DTI as string ranges: "<20%", "20%-<30%", etc.
# Convert to numeric midpoints. 43% is the QM (Qualified Mortgage) threshold —
# lenders face regulatory scrutiny above this, making it a natural cutpoint.
# This directly addresses Repo 1's limitation of missing debt controls.
# ==========================================================================
print("\n   ── NEW FEATURES ──────────────────────────────────────────────────")

# ── 9. DTI ENCODING (handles BOTH string ranges AND raw integers) ──────────
if 'debt_to_income_ratio' in df.columns:
    
    dti_map = {
        '<20%':      15.0,
        '20%-<30%':  25.0,
        '30%-<36%':  33.0,
        '36%-<37%':  36.5,
        '37%-<38%':  37.5,
        '38%-<39%':  38.5,
        '39%-<40%':  39.5,
        '40%-<42%':  41.0,
        '42%-<44%':  43.0,
        '44%-<46%':  45.0,
        '46%-<50%':  48.0,
        '50%-60%':   55.0,
        '>60%':      65.0,
        'Exempt':    np.nan,
        'NA':        np.nan,
    }

    def parse_dti(val):
        """
        Handles both HMDA DTI formats:
          - String ranges: '30%-<36%' → 33.0 (midpoint)
          - Raw integers:  '49'       → 49.0 (direct numeric)
          - Exempt / NA   → NaN
        """
        if pd.isna(val):
            return np.nan
        s = str(val).strip()
        # Try the string map first
        if s in dti_map:
            return dti_map[s]
        # Try parsing as a plain number (e.g. '49', '36')
        try:
            numeric = float(s)
            # HMDA integers are already percentages (e.g. 49 = 49%)
            # Clip to plausible range — values >100 or <0 are data errors
            if 0 <= numeric <= 100:
                return numeric
            else:
                return np.nan
        except (ValueError, TypeError):
            return np.nan

    # Vectorized DTI parsing — handles both string ranges AND raw integers
    # NEVER use .apply() on 43M rows — it's a Python loop and takes 45+ min
    
    dti_col = df['debt_to_income_ratio'].astype(str).str.strip()
    
    # Step 1: map known string ranges to midpoints
    dti_numeric = dti_col.map(dti_map)          # NaN for anything not in map
    
    # Step 2: for rows still NaN, try parsing as plain number (e.g. '49', '36')
    still_null  = dti_numeric.isnull()
    raw_numeric = pd.to_numeric(dti_col[still_null], errors='coerce')
    # Only accept values in plausible DTI range 0-100
    raw_numeric = raw_numeric.where((raw_numeric >= 0) & (raw_numeric <= 100))
    dti_numeric[still_null] = raw_numeric
    
    # Step 3: flag Exempt/NA explicitly as missing
    is_exempt = dti_col.isin(['Exempt', 'NA', 'nan', 'None', ''])
    dti_numeric[is_exempt] = np.nan
    
    df['dti_midpoint'] = dti_numeric.astype(np.float32)
    df['dti_missing']  = df['dti_midpoint'].isnull().astype(np.int8)
    df['dti_high']     = (df['dti_midpoint'] >= 43).astype(np.int8)
    median_dti         = df['dti_midpoint'].median()
    df['dti_midpoint'] = df['dti_midpoint'].fillna(median_dti)
    
    pct_present = 100 * (1 - df['dti_missing'].mean())
    
    
    print(f"   DTI encoding     : ✅  {pct_present:.1f}% of rows have DTI")
    
    print(f"     Median DTI              : {median_dti:.1f}%")
    print(f"     High DTI (≥43%)         : {100*df['dti_high'].mean():.1f}% of applications")
    print(f"     DTI missing/imputed     : {100*df['dti_missing'].mean():.1f}%")
else:
    df['dti_midpoint'] = np.float32(0)
    df['dti_high']     = np.int8(0)
    df['dti_missing']  = np.int8(1)
    print("   ⚠️  debt_to_income_ratio not found — DTI features set to 0")

# ── 10. CONSTRUCTION METHOD (NEW — fixes "property_type" naming issue) ────
# HMDA codes: 1=site-built, 2=manufactured home
# Manufactured home applicants face different underwriting rules
if 'construction_method' in df.columns:
    df['is_manufactured'] = (df['construction_method'] == 2).astype(np.int8)
    df['is_site_built']   = (df['construction_method'] == 1).astype(np.int8)
    print(f"   Construction method: ✅")
    print(f"     Site-built   : {100*df['is_site_built'].mean():.1f}%")
    print(f"     Manufactured : {100*df['is_manufactured'].mean():.1f}%")
else:
    df['is_manufactured'] = np.int8(0)
    df['is_site_built']   = np.int8(1)
    print("   ⚠️  construction_method not found — all assumed site-built")

# ── 11. AUS TYPE (NEW) ────────────────────────────────────────────────────
# Automated Underwriting System used by lender.
# aus_1 codes: 1=Fannie Mae DU, 2=Freddie Mac LPA, 3=TOTAL (FHA),
#              4=GUS (USDA), 5=Other, 6=Exempt, 1111=Not applicable
# KEY: Bartlett et al. (2022) show algorithmic lenders discriminate less.
# This variable lets us test that channel in our CATE model directly.
if 'aus_1' in df.columns:
    aus_numeric = pd.to_numeric(df['aus_1'], errors='coerce')
    df['aus_automated'] = aus_numeric.isin([1, 2, 3, 4]).astype(np.int8)
    df['aus_exempt']    = aus_numeric.isin([6, 1111]).astype(np.int8)
    df['aus_other']     = aus_numeric.isin([5]).astype(np.int8)
    print(f"   AUS type         : ✅")
    print(f"     Automated (DU/LPA/TOTAL/GUS): {100*df['aus_automated'].mean():.1f}%")
    print(f"     Exempt / not applicable     : {100*df['aus_exempt'].mean():.1f}%")
else:
    df['aus_automated'] = np.int8(0)
    df['aus_exempt']    = np.int8(0)
    df['aus_other']     = np.int8(0)
    print("   ⚠️  aus_1 not found — AUS features set to 0")

# Free raw string columns that are now fully encoded — reclaim ~1GB RAM
_drop = ['debt_to_income_ratio', 'combined_loan_to_value_ratio',
         'construction_method', 'aus_1', 'denial_reason_1',
         'tract_minority_population_percent', 'ffiec_msa_md_median_family_income']
df.drop(columns=[c for c in _drop if c in df.columns], inplace=True)
gc.collect()
print(f"   RAM after freeing raw cols: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")


# ── 12. NEIGHBOURHOOD RACIAL COMPOSITION (NEW) ────────────────────────────
# tract_minority_population_percent = % non-white residents in census tract.
# Tests whether the gap is worse for Black applicants in predominantly
# white neighbourhoods — a redlining-adjacent geographic mechanism.
if 'tract_minority_pct_fixed' in df.columns:
    df['tract_minority_pct'] = pd.to_numeric(
        df['tract_minority_pct_fixed'], errors='coerce'   # ← changed
    )
    tract_median             = df['tract_minority_pct'].median()
    df['tract_minority_pct'] = df['tract_minority_pct'].fillna(tract_median).astype(np.float32)
    q25 = df['tract_minority_pct'].quantile(0.25)
    q75 = df['tract_minority_pct'].quantile(0.75)
    df['high_minority_tract'] = (df['tract_minority_pct'] >= q75).astype(np.int8)
    df['low_minority_tract']  = (df['tract_minority_pct'] <= q25).astype(np.int8)
    print(f"   Tract minority % : ✅  median = {tract_median:.1f}%")
    print(f"     High minority tract (top 25%, ≥{q75:.1f}%)   : {100*df['high_minority_tract'].mean():.1f}% of apps")
    print(f"     Low minority tract (bottom 25%, ≤{q25:.1f}%) : {100*df['low_minority_tract'].mean():.1f}% of apps")
else:
    df['tract_minority_pct']  = np.float32(0)
    df['high_minority_tract'] = np.int8(0)
    df['low_minority_tract']  = np.int8(0)
    print("   ⚠️  tract_minority_population_percent not found")

# ── 13. APPLICANT AGE ENCODING (NEW) ──────────────────────────────────────
# HMDA reports age as string ranges: "25-34", "35-44", etc.
# Tests whether age × race interaction explains part of the gap
# (e.g., older Black applicants may have larger wealth gaps).
# ── 13. APPLICANT AGE ENCODING (NEW) ──────────────────────────────────────
if 'applicant_age' in df.columns:
    age_map = {
        '<25':   22.0, '25-34': 29.5, '35-44': 39.5,
        '45-54': 49.5, '55-64': 59.5, '65-74': 69.5,
        '>74':   77.0, '8888':  np.nan, '9999': np.nan,
    }
    # Force to plain numpy string first — avoids Arrow ExtensionArray overhead
    age_str = df['applicant_age'].to_numpy(dtype=str, na_value='9999')
    age_vals = np.array([age_map.get(v, np.nan) for v in age_str], dtype=np.float32)
    del age_str
    
    age_median = float(np.nanmedian(age_vals))
    age_vals[np.isnan(age_vals)] = age_median
    df['applicant_age_mid'] = age_vals
    del age_vals
    
    print(f"   Applicant age    : ✅  median = {age_median:.1f} years")
else:
    df['applicant_age_mid'] = np.float32(40.0)
    print("   ⚠️  applicant_age not found — set to 40.0")

# ── FINAL SUMMARY ──────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"✅ Feature construction complete")
print(f"   Dataset shape : {df.shape}")
print(f"   Total features built: {df.shape[1]} columns")
print(f"{'='*70}")

Loaded: 43,441,950 rows × 22 cols
RAM: 6.18 GB
FEATURE CONSTRUCTION
   ltv already present
   After LTV filter: 42,296,010 rows
   Loan purpose dummies: ✅
     Purchase : 17,974,926 (42.5%)
     Refi     : 17,558,917 (41.5%)
     HomeImprv: 3,207,169 (7.6%)
   Loan type dummies: ✅
   Occupancy dummies: ✅

   Computing lender size tiers...
     Small lenders (<500 apps)   :   201,248 applications
     Mid lenders (500-5000)      : 3,255,030 applications
     Large lenders (>5000 apps)  : 38,839,732 applications
   Log transforms: ✅
   Near PMI threshold (75-85% LTV): 7,557,110 applications

   ── NEW FEATURES ──────────────────────────────────────────────────
   DTI encoding     : ✅  98.2% of rows have DTI
     Median DTI              : 38.0%
     High DTI (≥43%)         : 32.1% of applications
     DTI missing/imputed     : 1.8%
   Construction method: ✅
     Site-built   : 96.1%
     Manufactured : 3.9%
   AUS type         : ✅
     Automated (DU/LPA/TOTAL/GUS): 67.0%
     Exempt / not

In [6]:
# ============================================================================
# CELL 6 — DEFINE FINAL FEATURE SETS
#
# We define three feature sets for use in downstream notebooks:
#   X_BASE   : matches Repo 1 controls (for ATE sanity check in NB19)
#   X_FULL   : all features for DML/CausalForest CATE estimation
#   X_HETERO : features used as effect modifiers in CATE visualisation
# ============================================================================

print("="*70)
print("DEFINING FEATURE SETS")
print("="*70)

# Base controls — identical to Repo 1 NB04 (for ATE validation)
X_BASE = [
    'income', 'loan_amount', 'property_value', 'ltv'
]

# Full feature set for CATE estimation
X_FULL = [
    # Continuous financials (original)
    'income', 'loan_amount', 'property_value', 'ltv',
    'log_income', 'log_loan_amount', 'log_property_val',

    # NEW: DTI (addresses Repo 1 missing-control concern directly)
    'dti_midpoint', 'dti_high', 'dti_missing',

    # Loan characteristics
    'purpose_purchase', 'purpose_refi', 'purpose_homeimprv',
    'type_conventional', 'type_fha', 'type_va', 'type_usda',
    'occ_primary', 'occ_second', 'occ_investment',

    # Lender
    'lender_small', 'lender_mid', 'lender_large',

    # NEW: Underwriting mechanism
    'aus_automated', 'aus_exempt',

    # NEW: Geography and neighbourhood
    'high_minority_tract', 'low_minority_tract', 'tract_minority_pct',

    # NEW: Applicant demographics
    'applicant_age_mid',

    # Threshold and time (original)
    'near_pmi_threshold', 'above_pmi', 'post_tightening', 'year'
]

# Effect modifiers — the dimensions along which we will slice CATE
X_HETERO = [
    'income', 'ltv', 'purpose_purchase', 'purpose_refi',
    'lender_small', 'lender_mid', 'lender_large',
    'post_tightening', 'near_pmi_threshold'
]

# Remove any features not present in df
X_BASE   = [f for f in X_BASE   if f in df.columns]
X_FULL   = [f for f in X_FULL   if f in df.columns]
X_HETERO = [f for f in X_HETERO if f in df.columns]

print(f"X_BASE   ({len(X_BASE)} features): {X_BASE}")
print(f"\nX_FULL   ({len(X_FULL)} features): {X_FULL}")
print(f"\nX_HETERO ({len(X_HETERO)} features): {X_HETERO}")

# Save feature set definitions for downstream notebooks
feature_sets = {
    'X_BASE': X_BASE,
    'X_FULL': X_FULL,
    'X_HETERO': X_HETERO
}

import json
with open(DATA_DIR / 'feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=2)
print("\n✅ Feature sets saved to data/feature_sets.json")

DEFINING FEATURE SETS
X_BASE   (4 features): ['income', 'loan_amount', 'property_value', 'ltv']

X_FULL   (33 features): ['income', 'loan_amount', 'property_value', 'ltv', 'log_income', 'log_loan_amount', 'log_property_val', 'dti_midpoint', 'dti_high', 'dti_missing', 'purpose_purchase', 'purpose_refi', 'purpose_homeimprv', 'type_conventional', 'type_fha', 'type_va', 'type_usda', 'occ_primary', 'occ_second', 'occ_investment', 'lender_small', 'lender_mid', 'lender_large', 'aus_automated', 'aus_exempt', 'high_minority_tract', 'low_minority_tract', 'tract_minority_pct', 'applicant_age_mid', 'near_pmi_threshold', 'above_pmi', 'post_tightening', 'year']

X_HETERO (9 features): ['income', 'ltv', 'purpose_purchase', 'purpose_refi', 'lender_small', 'lender_mid', 'lender_large', 'post_tightening', 'near_pmi_threshold']

✅ Feature sets saved to data/feature_sets.json


In [7]:
# ============================================================================
# CELL 7 — MISSINGNESS AUDIT
#
# Before saving, audit missingness in all features.
# We impute with medians for continuous, mode for categorical.
# Flag imputed rows with an indicator variable.
# Any feature with >20% missing is excluded from X_FULL.
# ============================================================================

print("="*70)
print("MISSINGNESS AUDIT")
print("="*70)

all_features = list(set(X_FULL + ['black', 'approved', 'year', 'lei']))
miss = df[all_features].isnull().sum().sort_values(ascending=False)
miss_pct = miss / len(df) * 100

miss_report = pd.DataFrame({
    'missing_count': miss,
    'missing_pct': miss_pct
}).query('missing_count > 0')

if len(miss_report) == 0:
    print("✅ No missing values in feature set")
else:
    print(miss_report.to_string())
    print()
    
    # Exclude features with >20% missing from X_FULL
    high_miss = miss_report[miss_report['missing_pct'] > 20].index.tolist()
    if high_miss:
        print(f"⚠️  Excluding from X_FULL (>20% missing): {high_miss}")
        X_FULL = [f for f in X_FULL if f not in high_miss]
    
    # Impute remaining missing with median/mode
    continuous_feats = ['income', 'loan_amount', 'property_value', 'ltv',
                        'log_income', 'log_loan_amount', 'log_property_val']
    
    for col in miss_report.index:
        if col in df.columns:
            if col in continuous_feats:
                fill_val = df[col].median()
                df[col] = df[col].fillna(fill_val)
                print(f"   Imputed {col} with median = {fill_val:.4f}")
            else:
                fill_val = df[col].mode().iloc[0] if len(df[col].mode()) > 0 else 0
                df[col] = df[col].fillna(fill_val)
                print(f"   Imputed {col} with mode = {fill_val}")

# Final check
remaining_miss = df[X_FULL + ['black', 'approved']].isnull().sum().sum()
print(f"\nRemaining nulls in feature + outcome columns: {remaining_miss}")
assert remaining_miss == 0, "Still have missing values — check imputation"

MISSINGNESS AUDIT
✅ No missing values in feature set

Remaining nulls in feature + outcome columns: 0


In [9]:
# ============================================================================
# CELL 8 — SAVE FEATURE MATRIX (BATCH WRITER — NO FULL COPY)
# Writes 5M rows at a time so peak RAM per batch is ~500MB not 6GB
# ============================================================================

import pyarrow as pa
import pyarrow.parquet as pq
import gc

print("=" * 70)
print("SAVING FEATURE MATRIX (BATCH WRITER)")
print("=" * 70)

# Define SAVE_COLS (was lost when Cell 8 was replaced)
SAVE_COLS = list(set(
    ['lei', 'year', 'applicant_race_1', 'black', 'approved'] +
    X_FULL + X_HETERO
))

# Only keep columns in SAVE_COLS
SAVE_COLS_PRESENT = [c for c in SAVE_COLS if c in df.columns]
missing_save = [c for c in SAVE_COLS if c not in df.columns]
if missing_save:
    print(f"⚠️  These SAVE_COLS not in df (will be skipped): {missing_save}")

print(f"Columns to save : {len(SAVE_COLS_PRESENT)}")
print(f"Rows to save    : {len(df):,}")

outpath    = DATA_DIR / 'features_panel.parquet'
BATCH_SIZE = 5_000_000  # 5M rows per batch — ~400-500MB each

writer  = None
n_total = len(df)

for start in range(0, n_total, BATCH_SIZE):
    end   = min(start + BATCH_SIZE, n_total)
    batch = df.iloc[start:end][SAVE_COLS_PRESENT]

    # Convert batch to PyArrow table
    table = pa.Table.from_pandas(batch, preserve_index=False)

    # Open writer on first batch (schema comes from first table)
    if writer is None:
        writer = pq.ParquetWriter(str(outpath), table.schema, compression='snappy')

    writer.write_table(table)

    del batch, table
    gc.collect()

    pct = 100 * end / n_total
    print(f"  Written rows {start:>10,} – {end:>10,}  ({pct:.0f}%)")

if writer:
    writer.close()

file_size_gb = outpath.stat().st_size / 1e9
print(f"\n✅ Saved successfully")
print(f"   Rows      : {n_total:,}")
print(f"   Columns   : {len(SAVE_COLS_PRESENT)}")
print(f"   File size : {file_size_gb:.2f} GB")

SAVING FEATURE MATRIX (BATCH WRITER)
Columns to save : 37
Rows to save    : 42,296,010
  Written rows          0 –  5,000,000  (12%)
  Written rows  5,000,000 – 10,000,000  (24%)
  Written rows 10,000,000 – 15,000,000  (35%)
  Written rows 15,000,000 – 20,000,000  (47%)
  Written rows 20,000,000 – 25,000,000  (59%)
  Written rows 25,000,000 – 30,000,000  (71%)
  Written rows 30,000,000 – 35,000,000  (83%)
  Written rows 35,000,000 – 40,000,000  (95%)
  Written rows 40,000,000 – 42,296,010  (100%)

✅ Saved successfully
   Rows      : 42,296,010
   Columns   : 37
   File size : 0.52 GB


In [10]:
# ============================================================================
# CELL 9 — FEATURE SUMMARY REPORT (POLARS — NO PANDAS LOAD)
# ============================================================================

import polars as pl

print("=" * 70)
print("FEATURE SUMMARY REPORT")
print("=" * 70)

fpath = DATA_DIR / 'features_panel.parquet'

# Use lazy scan — computes stats without loading full file into RAM
lf = pl.scan_parquet(str(fpath))
schema = lf.schema

# Compute all stats in one pass using Polars lazy evaluation
numeric_cols = [c for c, t in schema.items() if t in (
    pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64
)]
string_cols  = [c for c, t in schema.items() if c not in numeric_cols]

print(f"Numeric columns : {len(numeric_cols)}")
print(f"String/ID cols  : {string_cols}")

# Compute mean, std, min, max, null% for all numeric cols in one query
exprs = []
for c in numeric_cols:
    exprs += [
        pl.col(c).mean().alias(f"{c}__mean"),
        pl.col(c).std().alias(f"{c}__std"),
        pl.col(c).min().alias(f"{c}__min"),
        pl.col(c).max().alias(f"{c}__max"),
        (pl.col(c).is_null().sum() / pl.len() * 100).alias(f"{c}__null_pct"),
    ]

stats_row = lf.select(exprs).collect()

# Also get total rows
n_total = lf.select(pl.len()).collect().item()
print(f"Total rows      : {n_total:,}")

# Build summary table
summary_rows = []
for c in schema.keys():
    role = ('Treatment' if c == 'black' else
            'Outcome'   if c == 'approved' else
            'ID'        if c in ['lei', 'applicant_race_1'] else
            'Covariate')
    if c in numeric_cols:
        row = {
            'feature':     c,
            'role':        role,
            'dtype':       str(schema[c]),
            'mean':        round(stats_row[f"{c}__mean"][0], 4),
            'std':         round(stats_row[f"{c}__std"][0],  4),
            'min':         stats_row[f"{c}__min"][0],
            'max':         stats_row[f"{c}__max"][0],
            'missing_pct': round(stats_row[f"{c}__null_pct"][0], 2),
        }
    else:
        row = {'feature': c, 'role': role, 'dtype': str(schema[c]),
               'mean': None, 'std': None, 'min': None, 'max': None, 'missing_pct': 0}
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLES_DIR / 'nb17_feature_summary.csv', index=False)
print(summary_df[['feature','role','dtype','mean','std','missing_pct']].to_string(index=False))
print(f"\n✅ Saved to outputs/tables/nb17_feature_summary.csv")

# Black vs White breakdown — lazy, no full load
print("\n" + "=" * 70)
print("BLACK vs WHITE BY YEAR")
print("=" * 70)

breakdown = (
    lf.group_by(['year', 'black'])
    .agg([
        pl.len().alias('n'),
        pl.col('approved').mean().alias('approval_rate')
    ])
    .sort(['year', 'black'])
    .collect()
    .to_pandas()
)

for yr in sorted(breakdown['year'].unique()):
    yr_data = breakdown[breakdown['year'] == yr]
    w = yr_data[yr_data['black'] == 0].iloc[0]
    b = yr_data[yr_data['black'] == 1].iloc[0]
    gap = (w['approval_rate'] - b['approval_rate']) * 100
    b_share = 100 * b['n'] / (b['n'] + w['n'])
    print(f"  {yr}: Black={b['n']:>7,} ({b_share:.1f}%)  "
          f"White={w['n']:>9,}  Gap={gap:.2f}pp")

print(f"\n✅ NB17 Cell 9 complete — proceed to Cell 10")

FEATURE SUMMARY REPORT
Numeric columns : 36
String/ID cols  : ['lei']
Total rows      : 42,296,010
            feature      role   dtype        mean         std  missing_pct
 near_pmi_threshold Covariate    Int8      0.1787      0.3831       0.0000
       lender_small Covariate    Int8      0.0048      0.0688       0.0000
  applicant_age_mid Covariate Float32     47.1132     14.3843       0.0000
         aus_exempt Covariate    Int8      0.2917      0.4545       0.0000
                lei        ID  String         NaN         NaN       0.0000
   log_property_val Covariate Float32     12.7579      0.6872       0.0000
      aus_automated Covariate    Int8      0.6699      0.4703       0.0000
  purpose_homeimprv Covariate    Int8      0.0758      0.2647       0.0000
        occ_primary Covariate    Int8      0.9309      0.2536       0.0000
 tract_minority_pct Covariate Float32      0.0000      0.0000       0.0000
        loan_amount Covariate   Int32 261919.1782 352552.6983       0.0000
 

In [11]:
# ============================================================================
# CELL 10 — VERIFICATION CHECKS (POLARS LAZY — ZERO RAM LOAD)
# ============================================================================

import polars as pl
import gc

# Free df from memory first — we don't need it anymore
if 'df' in globals():
    del df
    gc.collect()
    print("✅ df freed from RAM")

outpath = DATA_DIR / 'features_panel.parquet'
lf = pl.scan_parquet(str(outpath))

print("=" * 70)
print("VERIFICATION CHECKS")
print("=" * 70)

# 1. Row count
n = lf.select(pl.len()).collect().item()
print(f"1. Total rows: {n:,}")
assert 40_000_000 <= n <= 45_000_000, f"Unexpected: {n:,}"
print(f"   ✅ Row count in expected range (40-45M)")

# 2. Black applicant share
black_share = lf.select(pl.col('black').mean()).collect().item()
print(f"2. Black applicant share: {100*black_share:.1f}%")
assert 0.08 <= black_share <= 0.14
print(f"   ✅ Black share in expected range (8-14%)")

# 3. Approval rate
appr_rate = lf.select(pl.col('approved').mean()).collect().item()
print(f"3. Overall approval rate: {100*appr_rate:.1f}%")
assert 0.75 <= appr_rate <= 0.90
print(f"   ✅ Approval rate in expected range (75-90%)")

# 4. Raw racial gap
white_appr = lf.filter(pl.col('black')==0).select(pl.col('approved').mean()).collect().item()
black_appr = lf.filter(pl.col('black')==1).select(pl.col('approved').mean()).collect().item()
raw_gap    = (white_appr - black_appr) * 100
print(f"4. Raw approval gap (White - Black): {raw_gap:.2f} pp")
assert 12 <= raw_gap <= 18, f"Gap out of range: {raw_gap:.2f}"
print(f"   ✅ Raw gap in expected range (12-18 pp) — consistent with Repo 1")

# 5. Nulls in key columns
key_cols   = ['black', 'approved', 'lei', 'year', 'ltv', 'income']
null_count = lf.select([pl.col(c).is_null().sum() for c in key_cols]).collect().to_pandas().sum().sum()
print(f"5. Null values in key columns: {null_count}")
assert null_count == 0
print(f"   ✅ No nulls in key columns")

# 6. LTV range
ltv_min = lf.select(pl.col('ltv').min()).collect().item()
ltv_max = lf.select(pl.col('ltv').max()).collect().item()
print(f"6. LTV range: [{ltv_min:.2f}, {ltv_max:.2f}]")
assert 1 <= ltv_min and ltv_max <= 200
print(f"   ✅ LTV range valid")

# 7. Years
years = sorted(lf.select(pl.col('year').unique()).collect()['year'].to_list())
print(f"7. Years present: {years}")
assert years == [2020, 2021, 2022, 2023, 2024]
print(f"   ✅ All 5 years present")

# 8. Columns present
cols = lf.columns
print(f"8. Columns ({len(cols)}): {cols}")

# 9. New features spot check
dti_pct  = lf.select((pl.col('dti_midpoint') > 0).mean()).collect().item()
aus_pct  = lf.select(pl.col('aus_automated').mean()).collect().item()
age_mean = lf.select(pl.col('applicant_age_mid').mean()).collect().item()
print(f"9. New feature spot checks:")
print(f"   DTI populated  : {100*dti_pct:.1f}%")
print(f"   AUS automated  : {100*aus_pct:.1f}%")
print(f"   Mean age       : {age_mean:.1f} years")
assert dti_pct > 0.90, "DTI mostly missing"
assert aus_pct > 0.50, "AUS mostly missing"
print(f"   ✅ New features populated correctly")

print("\n" + "=" * 70)
print("ALL VERIFICATION CHECKS PASSED")
print("features_panel.parquet is valid and ready.")
print("NB17 complete → proceed to NB18_overlap_diagnostics.ipynb")
print("=" * 70)

✅ df freed from RAM
VERIFICATION CHECKS
1. Total rows: 42,296,010
   ✅ Row count in expected range (40-45M)
2. Black applicant share: 9.9%
   ✅ Black share in expected range (8-14%)
3. Overall approval rate: 81.7%
   ✅ Approval rate in expected range (75-90%)
4. Raw approval gap (White - Black): 14.90 pp
   ✅ Raw gap in expected range (12-18 pp) — consistent with Repo 1
5. Null values in key columns: 0
   ✅ No nulls in key columns
6. LTV range: [1.00, 199.80]
   ✅ LTV range valid
7. Years present: [2020, 2021, 2022, 2023, 2024]
   ✅ All 5 years present
8. Columns (37): ['near_pmi_threshold', 'lender_small', 'applicant_age_mid', 'aus_exempt', 'lei', 'log_property_val', 'aus_automated', 'purpose_homeimprv', 'occ_primary', 'tract_minority_pct', 'loan_amount', 'dti_high', 'purpose_refi', 'above_pmi', 'type_usda', 'applicant_race_1', 'type_fha', 'post_tightening', 'dti_midpoint', 'purpose_purchase', 'type_va', 'ltv', 'low_minority_tract', 'income', 'property_value', 'black', 'high_minority_